# (2) Augmented SVR vs Non-Augmented SVR

- `SVR (no_aug)` vs `SVR (aug)`
- 5-Fold CV
- Save CSV results and print a comparison table

In [ ]:
import os
from datetime import datetime
from typing import List

import numpy as np
import pandas as pd
from tqdm import tqdm
from lazypredict.Supervised import LazyRegressor
from sklearn.model_selection import KFold
from sklearn.utils import all_estimators

RANDOM_STATE = 42
N_SPLITS = 5
EMBEDDINGS_DIR = "../Data"
OUT_DIR = "./lazy_regression_02_svr_noaug_vs_aug_result"
os.makedirs(OUT_DIR, exist_ok=True)

x_path = os.path.join(EMBEDDINGS_DIR, "MLM_train_aug_embeddings.npy")
y_path = os.path.join(EMBEDDINGS_DIR, "MLM_train_aug_labels.npy")

def compute_leaderboard_score(rmse: float, r2_value: float, y_range: float):
    normalized_rmse = rmse / y_range if y_range > 0 else np.inf
    r2_pearson = np.clip(r2_value, 0.0, 1.0)
    score = 0.4 * (1 - min(normalized_rmse, 1)) + 0.6 * r2_pearson
    return normalized_rmse, r2_pearson, score

def _find_first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"No available file found in candidates: {paths}")

def load_base_dataset():

    # Load data
    X = np.load(x_path)
    y = np.load(y_path)
    print(f"[LOAD] X={X.shape}, y={y.shape}")
    print(f"       X path: {x_path}")
    print(f"       y path: {y_path}")
    return X, y


from sklearn.svm import SVR
MODELS = [SVR]

print("[INFO] Final model count: 1")
print("[INFO] Included models:", [m.__name__ for m in MODELS])

def run_kfold_lazy(
    X: np.ndarray, y: np.ndarray,
    n_splits: int = 5, random_state: int = 42,
    out_dir: str = OUT_DIR,
):
    """Run 5-fold CV with a fixed model whitelist."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_results = []
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    y_full_range = float(y.max() - y.min())

    for fold_idx, (train_idx, valid_idx) in enumerate(
        kf.split(X), start=1
    ):
        X_train, X_valid = X[train_idx], X[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]

        reg = LazyRegressor(
            verbose=0,
            ignore_warnings=True,
            custom_metric=None,
            random_state=random_state,
            regressors=MODELS,
        )
        models, _ = reg.fit(X_train, X_valid, y_train, y_valid)
        models_df = models.reset_index().rename(columns={"index": "Model"})

        keep_cols = ["Model"]
        for col in ["Adjusted R-Squared", "R-Squared", "RMSE", "Time Taken"]:
            if col in models_df.columns:
                keep_cols.append(col)
        models_df = models_df[keep_cols]
        if "RMSE" in models_df.columns and "R-Squared" in models_df.columns:
            metrics = models_df.apply(
                lambda row: compute_leaderboard_score(
                    rmse=float(row["RMSE"]),
                    r2_value=float(row["R-Squared"]),
                    y_range=y_full_range,
                ),
                axis=1,
                result_type="expand",
            )
            metrics.columns = ["Normalized_RMSE", "R2_Pearson", "Leaderboard_Score"]
            models_df = pd.concat([models_df, metrics], axis=1)
        models_df.insert(0, "Data_Type", "MIN")
        models_df.insert(1, "Embedding_Type", 'MLM')
        models_df.insert(2, "Fold", fold_idx)

        fold_csv = os.path.join(
            out_dir,
            f"lazy_regressor_MIN_MLM_fold_{fold_idx}_{ts}.csv",
        )
        models_df.to_csv(fold_csv, index=False)
        fold_results.append(models_df)
        print(f"Fold {fold_idx}: saved {fold_csv}")

    all_results = pd.concat(fold_results, ignore_index=True)
    numeric_cols = all_results.select_dtypes(include=[np.number]).columns.tolist()
    if "Fold" in numeric_cols:
        numeric_cols.remove("Fold")

    mean_df = all_results.groupby("Model")[numeric_cols].mean().reset_index().add_prefix("mean_")
    std_df = all_results.groupby("Model")[numeric_cols].std(ddof=0).reset_index().add_prefix("std_")
    mean_df = mean_df.rename(columns={"mean_Model": "Model"})
    std_df = std_df.rename(columns={"std_Model": "Model"})
    overall_df = pd.merge(mean_df, std_df, on="Model", how="outer")
    
    if "mean_Leaderboard_Score" in overall_df.columns:
        overall_df = overall_df.sort_values("mean_Leaderboard_Score", ascending=False)
    elif "mean_RMSE" in overall_df.columns:
        overall_df = overall_df.sort_values("mean_RMSE", ascending=True)

    summary_path = os.path.join(
        out_dir,
        f"lazy_regressor_MIN_MLM_cv_summary_{ts}.csv",
    )
    overall_df.to_csv(summary_path, index=False)
    print("\nSummary:", summary_path)
    return overall_df, all_results, summary_path

In [ ]:
X, y = load_base_dataset()

In [ ]:
overall_df, all_results, summary_path = run_kfold_lazy(
    X, y, n_splits=5, random_state=42
)